# DMPBridge Block Comparison Notebook

This notebook compares two folders of JSON block files:

- Manual/reference blocks: `data/reference_structure_blocks`
- AI/Llama detected blocks: `data/llama_structured_blocks`

Expected file names:

```text
sample1_reference_blocks.json
sample2_reference_blocks.json
...

sample1_llama_blocks.json
sample2_llama_blocks.json
...
```

The notebook checks:

- whether each manual block was detected
- whether the label matched
- similarity between manual and AI text
- missing manual blocks
- extra AI-detected blocks
- summary statistics per sample

## Step 1 — Import libraries

In [1]:
from pathlib import Path
import json
import re
from difflib import SequenceMatcher
from collections import Counter
import pandas as pd

## Step 2 — Set project folders

In [2]:
# Detect project root
cwd = Path.cwd()

if (cwd / "data").exists() and (cwd / "src").exists():
    project_root = cwd
else:
    project_root = cwd.parent

# Your folders
reference_dir = project_root / "data" / "reference_structure_blocks"
detected_dir = project_root / "data" / "llama_structured_blocks"

# Output folder
output_dir = project_root / "data" / "block_comparison_reports"
output_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Reference folder:", reference_dir)
print("Detected folder:", detected_dir)
print("Output folder:", output_dir)

print("Reference folder exists:", reference_dir.exists())
print("Detected folder exists:", detected_dir.exists())

Project root: c:\Users\Nahid\dmpbridge
Reference folder: c:\Users\Nahid\dmpbridge\data\reference_structure_blocks
Detected folder: c:\Users\Nahid\dmpbridge\data\llama_structured_blocks
Output folder: c:\Users\Nahid\dmpbridge\data\block_comparison_reports
Reference folder exists: True
Detected folder exists: True


## Step 3 — Check file names

In [3]:
print("\nREFERENCE FILES")
for f in sorted(reference_dir.glob("*.json")):
    print(f.name)

print("\nAI / LLAMA FILES")
for f in sorted(detected_dir.glob("*.json")):
    print(f.name)


REFERENCE FILES
sample10_reference__blocks.json
sample1_reference_blocks.json
sample2_reference_blocks.json
sample3_reference_blocks.json
sample4_reference_blocks.json
sample5_refrence_blocks.json
sample6_reference_blocks.json
sample7_reference_blocks.json
sample8_refernce_blocks.json
sample9_reference_blocks.json

AI / LLAMA FILES
sample10_llama_blocks.json
sample1_llama_blocks.json
sample2_llama_blocks.json
sample3_llama_blocks.json
sample4_llama_blocks.json
sample5_llama_blocks.json
sample6_llama_blocks.json
sample7_llama_blocks.json
sample8_llama_blocks.json
sample9_llama_blocks.json


## Step 4 — Helper functions

In [4]:
def normalize_text(text):
    # Normalize text for fair comparison.
    # Original text is still preserved in the final reports.
    if text is None:
        return ""

    text = str(text)
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()


def get_sample_id(path):
    # Convert file names into the same sample ID.
    # sample1_reference_blocks.json -> sample1
    # sample1_llama_blocks.json     -> sample1

    stem = Path(path).stem.lower()
    stem = stem.replace("__", "_")

    suffixes = [
        "_reference_labels_blocks",
        "_reference_blocks",
        "_manual_labels_blocks",
        "_manual_blocks",
        "_llama_blocks",
        "_ai_blocks",
        "_detected_blocks",
        "_structured_blocks",
        "_blocks",
    ]

    for suffix in suffixes:
        if stem.endswith(suffix):
            stem = stem[:-len(suffix)]
            break

    stem = stem.replace("__", "_").strip("_")

    match = re.search(r"sample\s*[_-]*(\d+)", stem)
    if match:
        return f"sample{int(match.group(1))}"

    return stem


def load_json_blocks(path):
    # Load a JSON file and return a clean list of blocks.
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Support files where blocks are inside a dictionary.
    if isinstance(data, dict):
        for key in ["blocks", "structured_blocks", "data"]:
            if key in data and isinstance(data[key], list):
                data = data[key]
                break

    if not isinstance(data, list):
        raise ValueError(f"Expected list of blocks in {path}, but got {type(data)}")

    clean_blocks = []

    for i, block in enumerate(data):
        if not isinstance(block, dict):
            continue

        label = str(block.get("label", "")).strip()
        text = str(block.get("text", "")).strip()

        if not text:
            continue

        clean_blocks.append({
            "block_index": i + 1,
            "label": label,
            "text": text,
            "norm_text": normalize_text(text),
        })

    return clean_blocks


def text_similarity(a, b):
    # Character-level similarity.
    # Range: 0 to 1.
    a = normalize_text(a)
    b = normalize_text(b)

    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0

    return SequenceMatcher(None, a, b).ratio()


def token_set_similarity(a, b):
    # Word overlap similarity.
    # Useful when one block is split or merged.
    a_tokens = set(normalize_text(a).split())
    b_tokens = set(normalize_text(b).split())

    if not a_tokens and not b_tokens:
        return 1.0
    if not a_tokens or not b_tokens:
        return 0.0

    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)


def label_match(reference_label, detected_label):
    return str(reference_label).strip().lower() == str(detected_label).strip().lower()

## Step 5 — Verify sample ID matching

In [5]:
print("\nREFERENCE SAMPLE IDS")
for f in sorted(reference_dir.glob("*.json")):
    print(f.name, "->", get_sample_id(f))

print("\nAI / LLAMA SAMPLE IDS")
for f in sorted(detected_dir.glob("*.json")):
    print(f.name, "->", get_sample_id(f))


REFERENCE SAMPLE IDS
sample10_reference__blocks.json -> sample10
sample1_reference_blocks.json -> sample1
sample2_reference_blocks.json -> sample2
sample3_reference_blocks.json -> sample3
sample4_reference_blocks.json -> sample4
sample5_refrence_blocks.json -> sample5
sample6_reference_blocks.json -> sample6
sample7_reference_blocks.json -> sample7
sample8_refernce_blocks.json -> sample8
sample9_reference_blocks.json -> sample9

AI / LLAMA SAMPLE IDS
sample10_llama_blocks.json -> sample10
sample1_llama_blocks.json -> sample1
sample2_llama_blocks.json -> sample2
sample3_llama_blocks.json -> sample3
sample4_llama_blocks.json -> sample4
sample5_llama_blocks.json -> sample5
sample6_llama_blocks.json -> sample6
sample7_llama_blocks.json -> sample7
sample8_llama_blocks.json -> sample8
sample9_llama_blocks.json -> sample9


## Step 6 — Match reference files with detected files

In [6]:
reference_files = {
    get_sample_id(f): f
    for f in reference_dir.glob("*.json")
}

detected_files = {
    get_sample_id(f): f
    for f in detected_dir.glob("*.json")
}

matched_ids = sorted(
    set(reference_files.keys()) & set(detected_files.keys()),
    key=lambda x: int(re.search(r"\d+", x).group()) if re.search(r"\d+", x) else x
)

missing_detected = sorted(set(reference_files.keys()) - set(detected_files.keys()))
missing_reference = sorted(set(detected_files.keys()) - set(reference_files.keys()))

print("Number of reference files:", len(reference_files))
print("Number of detected files:", len(detected_files))
print("Number of matched file pairs:", len(matched_ids))
print("Matched IDs:", matched_ids)

if missing_detected:
    print("\nReference files without AI detected file:")
    print(missing_detected)

if missing_reference:
    print("\nAI detected files without reference file:")
    print(missing_reference)

Number of reference files: 10
Number of detected files: 10
Number of matched file pairs: 10
Matched IDs: ['sample1', 'sample2', 'sample3', 'sample4', 'sample5', 'sample6', 'sample7', 'sample8', 'sample9', 'sample10']


## Step 7 — Compare blocks inside one sample

In [7]:
def compare_blocks_for_sample(sample_id, reference_path, detected_path, similarity_threshold=0.70):
    # Compare one reference JSON file with one detected JSON file.

    reference_blocks = load_json_blocks(reference_path)
    detected_blocks = load_json_blocks(detected_path)

    reference_rows = []
    used_detected_indices = set()

    for ref in reference_blocks:
        best = None
        best_score = -1
        best_seq_score = 0
        best_token_score = 0

        for det in detected_blocks:
            seq_score = text_similarity(ref["text"], det["text"])
            token_score = token_set_similarity(ref["text"], det["text"])

            # Combined score helps when text is split or merged.
            combined_score = (seq_score * 0.60) + (token_score * 0.40)

            if combined_score > best_score:
                best_score = combined_score
                best = det
                best_seq_score = seq_score
                best_token_score = token_score

        if best is None:
            status = "missing"
            detected_index = None
            detected_label = ""
            detected_text = ""
            seq_score = 0
            token_score = 0
            combined_score = 0
            is_label_match = False
        else:
            detected_index = best["block_index"]
            detected_label = best["label"]
            detected_text = best["text"]
            seq_score = best_seq_score
            token_score = best_token_score
            combined_score = best_score
            is_label_match = label_match(ref["label"], detected_label)

            if combined_score >= similarity_threshold and is_label_match:
                status = "matched"
                used_detected_indices.add(detected_index)
            elif combined_score >= similarity_threshold and not is_label_match:
                status = "label_mismatch"
                used_detected_indices.add(detected_index)
            elif combined_score < similarity_threshold:
                status = "missing_or_weak_match"
            else:
                status = "needs_review"

        reference_rows.append({
            "sample_id": sample_id,
            "reference_file": reference_path.name,
            "detected_file": detected_path.name,
            "reference_block_index": ref["block_index"],
            "reference_label": ref["label"],
            "reference_text": ref["text"],
            "best_detected_block_index": detected_index,
            "best_detected_label": detected_label,
            "best_detected_text": detected_text,
            "sequence_similarity": round(seq_score, 4),
            "token_set_similarity": round(token_score, 4),
            "combined_similarity": round(combined_score, 4),
            "label_match": is_label_match,
            "status": status,
        })

    detected_rows = []

    for det in detected_blocks:
        if det["block_index"] not in used_detected_indices:
            detected_status = "extra_or_unmatched_ai_block"
        else:
            detected_status = "used_in_reference_match"

        detected_rows.append({
            "sample_id": sample_id,
            "detected_file": detected_path.name,
            "detected_block_index": det["block_index"],
            "detected_label": det["label"],
            "detected_text": det["text"],
            "status": detected_status,
        })

    status_counts = Counter(row["status"] for row in reference_rows)
    label_counts_ref = Counter(block["label"] for block in reference_blocks)
    label_counts_det = Counter(block["label"] for block in detected_blocks)

    matched_count = status_counts.get("matched", 0)
    label_mismatch_count = status_counts.get("label_mismatch", 0)
    weak_or_missing_count = status_counts.get("missing_or_weak_match", 0)

    summary = {
        "sample_id": sample_id,
        "reference_file": reference_path.name,
        "detected_file": detected_path.name,
        "reference_block_count": len(reference_blocks),
        "detected_block_count": len(detected_blocks),
        "matched_count": matched_count,
        "label_mismatch_count": label_mismatch_count,
        "missing_or_weak_match_count": weak_or_missing_count,
        "extra_ai_block_count": sum(1 for row in detected_rows if row["status"] == "extra_or_unmatched_ai_block"),
        "match_rate": round(matched_count / len(reference_blocks), 4) if reference_blocks else 0,
        "label_mismatch_rate": round(label_mismatch_count / len(reference_blocks), 4) if reference_blocks else 0,
        "reference_label_counts": dict(label_counts_ref),
        "detected_label_counts": dict(label_counts_det),
    }

    return summary, reference_rows, detected_rows

## Step 8 — Run comparison for all matched samples

In [8]:
all_summary_rows = []
all_reference_rows = []
all_detected_rows = []
full_json_report = {}

similarity_threshold = 0.70

for sample_id in matched_ids:
    ref_path = reference_files[sample_id]
    det_path = detected_files[sample_id]

    summary, reference_rows, detected_rows = compare_blocks_for_sample(
        sample_id=sample_id,
        reference_path=ref_path,
        detected_path=det_path,
        similarity_threshold=similarity_threshold,
    )

    all_summary_rows.append(summary)
    all_reference_rows.extend(reference_rows)
    all_detected_rows.extend(detected_rows)

    full_json_report[sample_id] = {
        "summary": summary,
        "reference_block_analysis": reference_rows,
        "detected_block_analysis": detected_rows,
    }

summary_df = pd.DataFrame(all_summary_rows)
reference_df = pd.DataFrame(all_reference_rows)
detected_df = pd.DataFrame(all_detected_rows)

print("Summary rows:", len(summary_df))
print("Reference block rows:", len(reference_df))
print("Detected block rows:", len(detected_df))

summary_df

Summary rows: 10
Reference block rows: 117
Detected block rows: 135


,sample_id,reference_file,detected_file,reference_block_count,detected_block_count,matched_count,label_mismatch_count,missing_or_weak_match_count,extra_ai_block_count,match_rate,label_mismatch_rate,reference_label_counts,detected_label_counts
0,sample1,sample1_reference_blocks.json,sample1_llama_blocks.json,26,23,16,4,6,3,0.6154,0.1538,"{'document_title': 1, 'section': 6, 'subsectio...","{'document_title': 1, 'section': 11, 'content'..."
1,sample2,sample2_reference_blocks.json,sample2_llama_blocks.json,12,24,5,2,5,17,0.4167,0.1667,"{'document_title': 1, 'section': 4, 'subsectio...","{'content': 12, 'section': 10, 'subsection': 2}"
2,sample3,sample3_reference_blocks.json,sample3_llama_blocks.json,14,9,8,0,6,1,0.5714,0.0000,"{'document_title': 1, 'section': 5, 'subsectio...","{'document_title': 1, 'content': 4, 'section': 4}"
3,sample4,sample4_reference_blocks.json,sample4_llama_blocks.json,2,2,2,0,0,0,1.0000,0.0000,"{'document_title': 1, 'content': 1}","{'document_title': 1, 'content': 1}"
4,sample5,sample5_refrence_blocks.json,sample5_llama_blocks.json,13,28,9,0,4,19,0.6923,0.0000,"{'document_title': 1, 'section': 6, 'content': 6}","{'document_title': 1, 'section': 10, 'content'..."
5,sample6,sample6_reference_blocks.json,sample6_llama_blocks.json,11,11,4,0,7,7,0.3636,0.0000,"{'document_title': 1, 'section': 5, 'content': 5}","{'document_title': 1, 'section': 5, 'content': 5}"
6,sample7,sample7_reference_blocks.json,sample7_llama_blocks.json,2,2,2,0,0,0,1.0000,0.0000,"{'document_title': 1, 'content': 1}","{'document_title': 1, 'content': 1}"
7,sample8,sample8_refernce_blocks.json,sample8_llama_blocks.json,13,13,13,0,0,0,1.0000,0.0000,"{'document_title': 1, 'section': 6, 'content': 6}","{'document_title': 1, 'section': 6, 'content': 6}"
8,sample9,sample9_reference_blocks.json,sample9_llama_blocks.json,11,10,10,0,1,0,0.9091,0.0000,"{'document_title': 1, 'section': 5, 'content': 5}","{'document_title': 1, 'content': 5, 'section': 4}"
9,sample10,sample10_reference__blocks.json,sample10_llama_blocks.json,13,13,13,0,0,0,1.0000,0.0000,"{'document_title': 1, 'section': 6, 'content': 6}","{'document_title': 1, 'section': 6, 'content': 6}"


## Step 9 — View problematic blocks

In [9]:
if len(reference_df) > 0:
    problems_df = reference_df[reference_df["status"] != "matched"].copy()
    print("Problematic reference blocks:", len(problems_df))
    display(problems_df[
        [
            "sample_id",
            "reference_block_index",
            "reference_label",
            "best_detected_label",
            "combined_similarity",
            "label_match",
            "status",
            "reference_text",
            "best_detected_text",
        ]
    ])
else:
    print("No reference comparison rows found.")

Problematic reference blocks: 35


,sample_id,reference_block_index,reference_label,best_detected_label,combined_similarity,label_match,status,reference_text,best_detected_text
2,sample1,3,subsection,section,0.4141,False,missing_or_weak_match,A. Types and amount of scientific data expecte...,C. When and how long the scientific data will ...
4,sample1,5,subsection,section,1.0000,False,label_mismatch,B. Scientific data that will be preserved and ...,B. Scientific data that will be preserved and ...
6,sample1,7,subsection,section,1.0000,False,label_mismatch,"C. Metadata, other relevant data, and associat...","C. Metadata, other relevant data, and associat..."
13,sample1,14,subsection,section,0.4608,False,missing_or_weak_match,A. Repository where scientific data and metada...,B. Whether access to scientific data will be c...
14,sample1,15,content,content,0.3738,True,missing_or_weak_match,De-identified data will be uploaded to the Uni...,A. Repository where scientific data and metada...
15,sample1,16,subsection,section,0.5567,False,missing_or_weak_match,B. How scientific data will be findable and id...,C. When and how long the scientific data will ...
16,sample1,17,content,content,0.6639,True,missing_or_weak_match,Data will be findable via publications and on ...,"In addition to the data described above, code ..."
17,sample1,18,subsection,section,1.0000,False,label_mismatch,C. When and how long the scientific data will ...,C. When and how long the scientific data will ...
20,sample1,21,subsection,section,0.4941,False,missing_or_weak_match,"A. Factors affecting subsequent access, distri...","Element 5: Access, Distribution, or Reuse Cons..."
22,sample1,23,subsection,section,1.0000,False,label_mismatch,B. Whether access to scientific data will be c...,B. Whether access to scientific data will be c...


## Step 10 — View extra AI blocks

In [10]:
if len(detected_df) > 0:
    extra_ai_df = detected_df[detected_df["status"] == "extra_or_unmatched_ai_block"].copy()
    print("Extra/unmatched AI blocks:", len(extra_ai_df))
    display(extra_ai_df[
        [
            "sample_id",
            "detected_block_index",
            "detected_label",
            "status",
            "detected_text",
        ]
    ])
else:
    print("No detected comparison rows found.")

Extra/unmatched AI blocks: 47


,sample_id,detected_block_index,detected_label,status,detected_text
12,sample1,13,content,extra_or_unmatched_ai_block,A. Repository where scientific data and metada...
19,sample1,20,section,extra_or_unmatched_ai_block,"Protections for privacy, rights, and confident..."
20,sample1,21,content,extra_or_unmatched_ai_block,All data used in this study was de-identified ...
25,sample2,3,content,extra_or_unmatched_ai_block,Data management plans should describe whether ...
26,sample2,4,subsection,extra_or_unmatched_ai_block,"Data Types and Sources. A brief, high-level de..."
27,sample2,5,content,extra_or_unmatched_ai_block,the course of the proposed research and which ...
28,sample2,6,section,extra_or_unmatched_ai_block,Data Sharing and Data Preservation. A descript...
29,sample2,7,subsection,extra_or_unmatched_ai_block,"This should include, where appropriate:"
30,sample2,8,content,extra_or_unmatched_ai_block,the anticipated means for sharing and rational...
31,sample2,9,section,extra_or_unmatched_ai_block,Data will also be deposited into Northwestern ...


## Step 11 — Save reports

In [11]:
summary_csv = output_dir / "summary_report.csv"
reference_csv = output_dir / "reference_block_level_report.csv"
detected_csv = output_dir / "detected_ai_block_level_report.csv"
json_report_path = output_dir / "full_block_comparison_report.json"

summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
reference_df.to_csv(reference_csv, index=False, encoding="utf-8-sig")
detected_df.to_csv(detected_csv, index=False, encoding="utf-8-sig")

with open(json_report_path, "w", encoding="utf-8") as f:
    json.dump(full_json_report, f, indent=2, ensure_ascii=False)

print("Saved:")
print(summary_csv)
print(reference_csv)
print(detected_csv)
print(json_report_path)

Saved:
c:\Users\Nahid\dmpbridge\data\block_comparison_reports\summary_report.csv
c:\Users\Nahid\dmpbridge\data\block_comparison_reports\reference_block_level_report.csv
c:\Users\Nahid\dmpbridge\data\block_comparison_reports\detected_ai_block_level_report.csv
c:\Users\Nahid\dmpbridge\data\block_comparison_reports\full_block_comparison_report.json


## Step 12 — Overall interpretation

In [12]:
if len(summary_df) > 0:
    print("Overall results")
    print("----------------")
    print("Total samples:", len(summary_df))
    print("Total reference blocks:", summary_df["reference_block_count"].sum())
    print("Total detected blocks:", summary_df["detected_block_count"].sum())
    print("Total matched blocks:", summary_df["matched_count"].sum())
    print("Total label mismatches:", summary_df["label_mismatch_count"].sum())
    print("Total missing/weak matches:", summary_df["missing_or_weak_match_count"].sum())
    print("Total extra AI blocks:", summary_df["extra_ai_block_count"].sum())
    print("Average match rate:", round(summary_df["match_rate"].mean(), 4))

    display(summary_df[
        [
            "sample_id",
            "reference_block_count",
            "detected_block_count",
            "matched_count",
            "label_mismatch_count",
            "missing_or_weak_match_count",
            "extra_ai_block_count",
            "match_rate",
        ]
    ])
else:
    print("No matched samples were found. Check folder paths and file names.")

Overall results
----------------
Total samples: 10
Total reference blocks: 117
Total detected blocks: 135
Total matched blocks: 82
Total label mismatches: 6
Total missing/weak matches: 29
Total extra AI blocks: 47
Average match rate: 0.7568


,sample_id,reference_block_count,detected_block_count,matched_count,label_mismatch_count,missing_or_weak_match_count,extra_ai_block_count,match_rate
0,sample1,26,23,16,4,6,3,0.6154
1,sample2,12,24,5,2,5,17,0.4167
2,sample3,14,9,8,0,6,1,0.5714
3,sample4,2,2,2,0,0,0,1.0000
4,sample5,13,28,9,0,4,19,0.6923
5,sample6,11,11,4,0,7,7,0.3636
6,sample7,2,2,2,0,0,0,1.0000
7,sample8,13,13,13,0,0,0,1.0000
8,sample9,11,10,10,0,1,0,0.9091
9,sample10,13,13,13,0,0,0,1.0000
